# Rectified Flow Inversion — Stable Audio 3

RF-Inversion ([Rout et al., 2024](https://arxiv.org/abs/2410.10792)) on
[`stabilityai/stable-audio-3-medium-base`](https://huggingface.co/stabilityai/stable-audio-3-medium-base) —
the un-post-trained rectified-flow checkpoint, which is the one that inverts cleanly.

The flow ODE runs `t = 1` (noise) → `t = 0` (data). Inversion is the same ODE integrated the
other way. The two optional controllers (`gamma`, `eta`) are linear blends in velocity space;
with both at `0` you get plain deterministic inversion, which should reconstruct the input.

Implementation lives in `rf_inversion.py`.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "src")   # implementations live in src/

import torch
import torchaudio
import IPython.display as ipd

from stable_audio_3.model import StableAudioModel
from stable_audio_3.inference.audio_utils import prepare_audio

from rf_inversion import invert, sample, invert_and_edit, make_cond

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)


# "medium-base"      1.4B, ~9.2 GB fp32, best quality
# "small-music-base" 433M, ~2.1 GB, music only, several times faster
# "small-sfx-base"   433M, ~2.1 GB, sound effects only
# Use a *-base checkpoint: the post-trained ones are distilled for few-step
# ping-pong sampling and do not invert cleanly.
MODEL_ID = "small-music-base"

model = StableAudioModel.from_pretrained(MODEL_ID, device=device)

SR = model.model.sample_rate                  # 44100
DS = model.same.downsampling_ratio            # 4096 -> ~10.8 latent frames/sec

print(f"{MODEL_ID}: {sum(p.numel() for p in model.model.model.parameters()) / 1e6:.0f}M params")
print("objective:", model.model.diffusion_objective)
print("sample rate:", SR, "| downsampling:", DS, "| latent channels:", model.model.io_channels)


AUDIO_DIR = Path("audio")          # sources live here; generated clips are written here too
AUDIO_DIR.mkdir(exist_ok=True)


def load_latent(path, seconds=10.0, offset=0.0):
    """Load audio -> normalised stereo tensor + latent.

    `seconds` is a maximum: a shorter file keeps its own length rather than being
    padded out with silence. Length is snapped down to a multiple of DS.
    """
    wav, sr = torchaudio.load(AUDIO_DIR / path)
    wav = wav[:, int(offset * sr):]
    available = int(wav.shape[-1] / sr * SR)          # length once resampled to SR
    n = max((min(int(seconds * SR), available) // DS) * DS, DS)
    audio = prepare_audio(wav, in_sr=sr, target_sr=SR, target_length=n, target_channels=2, device=device)
    audio = audio / audio.abs().max().clamp(min=1e-6)
    latent = model.same.encode(audio.to(next(model.same.parameters()).dtype))
    return audio, latent, n / SR


def to_audio(latent):
    return model.same.decode(latent).float().clamp(-1, 1).cpu()


def play(audio, label=None, name=None):
    """`name` is a bare filename; everything is written under AUDIO_DIR."""
    audio = audio[0] if audio.dim() == 3 else audio
    audio = (audio / audio.abs().max().clamp(min=1e-6)).float().cpu()
    if name:
        torchaudio.save(str(AUDIO_DIR / name), audio, SR)
    if label:
        print(label)
    ipd.display(ipd.Audio(audio.numpy(), rate=SR))
    

# Solver and schedule are a matched pair (section 2). Swap both together:
#   ("midpoint", "logsnr")     default, cheapest and most accurate
#   ("fixed-point", "model")   the older pairing, needs FP_ITERS
STEPS = 50                                # SA3 -base default; 8 for distilled ckpts
SOLVER, SCHEDULE = "midpoint", "logsnr"
SAMPLE_SOLVER = "euler" if SOLVER == "fixed-point" else SOLVER
FP_ITERS = 2                              # fixed-point only; ignored by midpoint

NFE = [0]                                 # DiT call counter; cells reset it themselves
try:
    _nfe_handle.remove()
except NameError:
    pass
_nfe_handle = model.model.model.register_forward_pre_hook(
    lambda *_: NFE.__setitem__(0, NFE[0] + 1))

print(f"{STEPS} steps | {SOLVER} + {SCHEDULE}")


## 1. Autoencoder round-trip

Before blaming the inversion for anything, check what the SAME autoencoder alone gives back.
This is the ceiling on reconstruction quality.

In [ ]:
AUDIO_PATH, N_SEC = "loop.wav", 10.0     # source clip for this cell
audio, latent, seconds_total = load_latent(AUDIO_PATH, seconds=N_SEC)

play(to_audio(latent), "Autoencoder round-trip:", "ae_roundtrip.wav")

## 2. Inversion sanity check

`gamma = 0`, `eta = 0`, empty prompt, no CFG — deterministic inversion followed by deterministic
re-sampling over the same schedule.

Solver and schedule are one choice, not two: both passes must discretise the ODE the same way or
the round trip does not close, and the two solvers want opposite spacings. Round trip on 10s of
rain, 50 steps:

| solver | schedule | DiT calls | `small-music-base` | `medium-base` |
|---|---|---|---|---|
| **`midpoint`** | **`logsnr`** | **102** | **0.0033** | **0.0044** |
| `midpoint` | `model` | 102 | 0.0282 | 0.0054 |
| `fixed-point` | `model` | 200 | 0.0187 | 0.0247 |

`fixed_point_iters` belongs to the fixed-point pairing only.


In [ ]:
AUDIO_PATH, N_SEC = "loop.wav", 10.0     # source clip for this cell
audio, latent, seconds_total = load_latent(AUDIO_PATH, seconds=N_SEC)

cond = make_cond(model, "", seconds_total, latent.shape[-1])

inverted = invert(model, latent, cond, steps=STEPS, gamma=0.0,
                  solver=SOLVER, schedule=SCHEDULE, fixed_point_iters=FP_ITERS)
recon = sample(model, inverted, cond, steps=STEPS, cfg_scale=1.0, eta=0.0,
               solver=SAMPLE_SOLVER, schedule=SCHEDULE)

rel_err = ((recon - latent).norm() / latent.norm()).item()
print(f"inverted noise: mean {inverted.mean():+.3f}  std {inverted.std():.3f}  (N(0,1) if it were pure noise)")
print(f"latent rel-err: {rel_err:.4f}")   # ~0.003 small-music-base, ~0.004 medium-base

play(to_audio(recon), "Reconstruction:", "reconstructed.wav")
play(audio, "Reference (for comparison):")

## 3. Editing

Invert under the source prompt, re-sample under the target.

Inside the window each sampling step is a mix of two velocities: the model's, following the target
prompt, and a closed-form one pointing straight at the input latent — `(x - x_0) / t`, no prompt
and no model call. Outside the window only the model's is used. The source prompt plays no part
here; it only shapes the inversion, so it decides which noise the run starts from and nothing else.

| knob | paper | what it does |
|---|---|---|
| `ETA` | η | the mix inside the window. 0 = target prompt only, 1 = the line to the input latent only, which ignores the model and so the prompt |
| `START`, `STOP` | s, τ | the fractions of the run, counted from the noise end, between which that mix applies. Outside them the target prompt is followed alone, so `ETA=1` returns the input only when `START=0` `STOP=1` |
| `GAMMA` | γ | the same construction on the inversion pass, mixing in a line toward a Gaussian sample. 1 guarantees the inverted latent is Gaussian, 0 reconstructs best |

RF-Inversion used `START` 0.0 and `STOP` 0.3, and reported `STOP` past 0.14 already dominating the
step.

In [ ]:
AUDIO_PATH, N_SEC = "loop.wav", 10.0     # source clip for this cell
audio, latent, seconds_total = load_latent(AUDIO_PATH, seconds=N_SEC)

SOURCE_PROMPT = ""      # empty = unguided inversion
PROMPT = "pots and pans"
CFG = 7.0               # SA3 -base default 7.0
ETAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # 0-1
START, STOP = 0.0, 0.3  # 0-1 each; paper 0.0 -> 0.3, paper reports saturating past 0.14
                        # ETA=1 returns the input exactly, but only at STOP=1.0

# secondary
GAMMA = 0.0             # 0-1
APG = 1.0               # 0-1; SA3 default 1.0
NORM_MATCH = False      # not in the paper

src_cond = make_cond(model, SOURCE_PROMPT, seconds_total, latent.shape[-1])
edit_cond = make_cond(model, PROMPT, seconds_total, latent.shape[-1])

inverted = invert(model, latent, src_cond, steps=STEPS, gamma=GAMMA, norm_match=NORM_MATCH,
                  solver=SOLVER, schedule=SCHEDULE, fixed_point_iters=FP_ITERS)

for eta in ETAS:
    out = sample(
        model, inverted, edit_cond,
        steps=STEPS, cfg_scale=CFG, apg_scale=APG, eta=eta, source_latent=latent,
        start=START, stop=STOP, norm_match=NORM_MATCH,
        solver=SAMPLE_SOLVER, schedule=SCHEDULE,
    )
    play(to_audio(out), f"eta={eta}  cfg={CFG}", f"sweep_eta{eta}.wav")

## 4. FlowEdit — editing without inverting

[FlowEdit](https://arxiv.org/abs/2412.08629) never inverts. It runs one ODE from the input clip to
the edited one, driven by the difference between what the model predicts under each prompt:

```
eps ~ N(0, I)                          fresh every step
z_src = (1 - t) * x_src + t * eps      a point on the input's own noising path
z_tgt = z_src + delta                  the same point, carrying the edit so far
delta += dt * (v(z_tgt, t | target) - v(z_src, t | source))
```

`delta` starts at zero, so the output starts as the input and moves only as far as the two prompts
disagree. Anything the model would predict for both cancels in the subtraction, so it is never
regenerated. With the same prompt on both sides the output is the input exactly.

| knob | paper | what it does |
|---|---|---|
| `T_STARTS` | n_max | the edit only accumulates below this `t`, so a higher value spends more of the run editing and changes more. 1.0 uses the whole run |
| `T_STOP` | n_min | below this `t`, stop editing and sample the target prompt normally. Added by the paper for style changes; keeps less of the input |
| `N_AVG` | n_avg | how many noise draws are averaged per step. Fresh noise is drawn every step, so this is the only control over how much the result varies between seeds |
| `SRC_CFG`, `TGT_CFG` | — | guidance for each prompt. Only the difference between them drives the edit, since a shared component cancels along with everything else |

In [ ]:
from flow_edit import flow_edit

AUDIO_PATH, N_SEC = "loop.wav", 10.0     # source clip for this cell
audio, latent, seconds_total = load_latent(AUDIO_PATH, seconds=N_SEC)

SOURCE_PROMPT = "drum loop"
PROMPT = "pots and pans"
SRC_CFG, TGT_CFG = 1.8, 7.0   # target = SA3's 7.0; source = 7.0 * the paper's src:tgt ratio
                              # (3.5/13.5 on SD3, 1.5/5.5 on FLUX, both ~0.26)
T_STARTS = [0.9, 0.93, 0.96, 1.0]   # 0-1; paper edits over the last 66% (SD3) / 86% (FLUX) of steps

# secondary
T_STOP = 0.0      # 0-1; paper 0
N_AVG = 1         # paper 1; each step costs 2 model calls
SEED = 0
APG = 1.0         # 0-1; SA3 default 1.0

for t_start in T_STARTS:
    out = flow_edit(
        model, latent,
        target_prompt=PROMPT,
        seconds_total=seconds_total,
        source_prompt=SOURCE_PROMPT,
        steps=STEPS,
        t_start=t_start,
        t_stop=T_STOP,
        n_avg=N_AVG,
        src_cfg=SRC_CFG,
        tgt_cfg=TGT_CFG,
        apg_scale=APG,
        schedule="model",
        seed=SEED,
    )
    drift = ((out - latent).norm() / latent.norm()).item()
    play(to_audio(out), f"t_start={t_start}  drift={drift:.3f}", f"flowedit_t{t_start}.wav")

## 5. Attention injection

[FireFlow](https://arxiv.org/abs/2412.07517)'s editing method. In self-attention,
`out = softmax(q kᵀ/√d) v`, the values `v` are what each position reads and `q`, `k` decide where
it reads from. The inversion pass records `v` for every block and step; the sampling pass writes
those recordings back in under the new prompt. So the new prompt still decides where each position
looks, but what it finds there comes from the input clip. Costs no extra model calls.

In images this pins layout, because self-attention there runs over space. Here the sequence is 64
non-audio tokens followed by one per audio frame (~10.8/sec), so attention runs over time and what
gets pinned is timing.

| knob | FireFlow | what it does |
|---|---|---|
| `T_STARTS` | inject | the recording is replayed while `t` is above this, so a lower value replays over more of the run and stays closer to the input |
| `FEATURE` | qkv_ratio | which tensors come from the input: `"v"` (what each position reads) or `"kv"` (also where it reads from) |
| `CFG_HALF` | — | the prompted and unprompted passes run as one batch; this picks which gets the recording. No equivalent in the paper, since Flux has no unprompted pass |
| `LAYERS` | start/end_layer_index | which of the 20 transformer blocks replay (20 on small, 24 on medium) |
| `STRENGTH` | — | 1 uses the recording, 0 ignores it, in between mixes the two |
| `SCOPE` | — | whether the 64 non-audio tokens are replayed along with the audio ones |

In [ ]:
from attn_inject import attn_edit

AUDIO_PATH, N_SEC = "loop.wav", 10.0     # source clip for this cell
audio, latent, seconds_total = load_latent(AUDIO_PATH, seconds=N_SEC)

SOURCE_PROMPT = "drum loop"
PROMPT = "pots and pans"
CFG = 7.0                                   # SA3 -base default 7.0
T_STARTS = [0.96, 0.90, 0.85, 0.78, 0.68]   # 0-1. Paper replayed over its first 20 of 25
                                            # steps = T_START ~ 0.03 on the logsnr schedule

# secondary
FEATURE = "kv"      # "v" | "kv"; paper "v"
CFG_HALF = "cond"   # "cond" | "both" | "uncond"; no paper equivalent
LAYERS = None       # None = all 20 (24 on medium-base), or a pair like (7, 13)
                    # paper used blocks 20-37 of Flux's 57, i.e. (7, 13) at this depth
STRENGTH = 1.0      # 0-1; paper 1.0 (its qkv_ratio)
SCOPE = "all"       # "all" | "audio"
ETA = 0.0           # 0-1; section 3's controller, usable at the same time

for t_start in T_STARTS:
    out, _ = attn_edit(
        model, latent, target_prompt=PROMPT, seconds_total=seconds_total,
        source_prompt=SOURCE_PROMPT, steps=STEPS, t_start=t_start,
        feature=FEATURE, cfg_half=CFG_HALF, layers=LAYERS,
        strength=STRENGTH, scope=SCOPE,
        eta=ETA, cfg_scale=CFG, seed=0,
    )
    drift = ((out - latent).norm() / latent.norm()).item()
    play(to_audio(out), f"t_start={t_start}  drift={drift:.3f}", f"inject_t{t_start}.wav")